# LLM-as-a-Judge Accuracy Evaluation: HRBench 8K Benchmark
## Double-Blind Option Matching via Local Qwen/Qwen2.5-7B-Instruct

> **Objective**: Automatically evaluate and score free-form Vision-Language Model predictions from `mssearch_hrbench8k_predictions.jsonl` using a local **`Qwen/Qwen2.5-7B-Instruct`** model on **Kaggle GPU (2x T4 or 1x P100)** with **Zero External API Costs**.

### 👑 Double-Blind Evaluation Protocol (Zero Selection Bias)
- The Judge LLM is provided only with: `<question>`, `<options>` (A, B, C, D), and `<model_prediction>`.
- **Zero Ground-Truth Leakage**: The Ground Truth answer letter is completely withheld from the judge prompt to prevent position sycophancy or bias.
- **Rigorous Output Validation**: Judge outputs strict JSON: `{"chosen_option": "A", "confidence": 1.0, "reasoning": "..."}`.
- If the model prediction contradicts all candidate options or is unanswerable, the judge outputs `None` (scored as incorrect).
- Accuracy is computed per category (`single`, `cross`, `overall`).
- Incremental checkpointing is supported to `/kaggle/working/answers/mssearch_hrbench8k_judged.jsonl`.


In [ ]:
!nvidia-smi -L

import os
import sys
import gc
import json
import re
import time
import torch
import pandas as pd
from collections import Counter, defaultdict
from tqdm.auto import tqdm

print(f"PyTorch Version : {torch.__version__}")
print(f"CUDA Available  : {torch.cuda.is_available()}")
print(f"GPU Device Count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"  • GPU {i}: {props.name} ({props.total_memory / (1024**3):.1f} GB VRAM)")

# Install transformers and accelerate if needed
!pip install -q "transformers>=4.45.0" "accelerate>=0.26.0" "tabulate"
print("[SETUP] Environment verified and packages ready!")


In [ ]:
# ==============================================================================
# EVALUATION PATHS & CONFIGURATION
# ==============================================================================

# Path to predictions file generated by runner notebook
PREDICTIONS_INPUT_PATH = "/kaggle/working/answers/mssearch_hrbench8k_predictions.jsonl"

# Fallback search across common Kaggle input directories
if not os.path.exists(PREDICTIONS_INPUT_PATH):
    candidates = [
        "/kaggle/working/mssearch_hrbench8k_predictions.jsonl",
        "/kaggle/working/answers/mssearch_track_a_hrbench.jsonl",
        "/kaggle/input/mssearch-predictions/mssearch_hrbench8k_predictions.jsonl",
        "docs/experiments/freeform/results/not_freeform/cvsearch_hrbench8k.jsonl"
    ]
    for c in candidates:
        if os.path.exists(c):
            PREDICTIONS_INPUT_PATH = c
            break

JUDGE_MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
OUTPUT_DIR = "/kaggle/working/answers"
os.makedirs(OUTPUT_DIR, exist_ok=True)
JUDGED_OUTPUT_JSONL = os.path.join(OUTPUT_DIR, "mssearch_hrbench8k_judged.jsonl")
WRONG_SAMPLES_JSONL = os.path.join(OUTPUT_DIR, "mssearch_hrbench8k_wrong_samples.jsonl")
WRONG_SAMPLES_JSON = os.path.join(OUTPUT_DIR, "mssearch_hrbench8k_wrong_samples.json")

print(f"[CONFIG] Input Predictions File : {PREDICTIONS_INPUT_PATH} (Exists: {os.path.exists(PREDICTIONS_INPUT_PATH)})")
print(f"[CONFIG] Local Judge Model ID   : {JUDGE_MODEL_ID}")
print(f"[CONFIG] Judged Output File     : {JUDGED_OUTPUT_JSONL}")
print(f"[CONFIG] Wrong Samples JSONL   : {WRONG_SAMPLES_JSONL}")
print(f"[CONFIG] Wrong Samples JSON    : {WRONG_SAMPLES_JSON}")

In [ ]:
print("=" * 75)
print(f"LOADING LOCAL JUDGE MODEL: {JUDGE_MODEL_ID}")
print("=" * 75)

from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(JUDGE_MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    JUDGE_MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)
model.eval()

print(f"[SUCCESS] {JUDGE_MODEL_ID} loaded successfully across devices:")
for i in range(torch.cuda.device_count()):
    alloc = torch.cuda.memory_allocated(i) / (1024**3)
    print(f"  • GPU {i} VRAM Allocated: {alloc:.2f} GB")
print("=" * 75)


In [ ]:
DOUBLE_BLIND_SYSTEM_PROMPT = """You are an expert, objective benchmark evaluator for Vision-Language Models.
Your task is to analyze a model's free-form natural language response to a visual multiple-choice question and determine which candidate option (A, B, C, D, or None) the model is asserting or choosing.

Evaluation Rules:
1. Identify the core factual claim in the model's prediction. Ignore polite conversational filler (e.g., "The image shows...", "In my opinion...", "It appears to be...").
2. Resolve linguistic synonyms, number words ("two" == "2", "four" == "4"), demonyms ("German" == "Germany"), diacritics, and spatial relations.
3. If the prediction clearly aligns with one option, output that option's letter ('A', 'B', 'C', or 'D').
4. If the prediction contradicts all candidate options, describes something completely different, or states it cannot see the object, output 'None'.
5. You MUST respond ONLY with a valid JSON object conforming to the schema below."""

DOUBLE_BLIND_USER_TEMPLATE = """<question>
{question}
</question>

<options>
{options_formatted}
</options>

<model_prediction>
{prediction}
</model_prediction>

Respond with JSON in this exact structure:
{{
  "chosen_option": "A",
  "confidence": 1.0,
  "reasoning": "Brief 1-2 sentence rationale explaining the matching decision."
}}"""

def format_options(options_obj, cycle_idx=0):
    if isinstance(options_obj, dict):
        lines = [f"{k}: {v}" for k, v in sorted(options_obj.items())]
        return "\n".join(lines)
    elif isinstance(options_obj, list):
        if len(options_obj) > 0 and isinstance(options_obj[0], str) and ('\n' in options_obj[0] or 'A.' in options_obj[0] or 'A:' in options_obj[0]):
            idx = cycle_idx if (0 <= cycle_idx < len(options_obj)) else 0
            return options_obj[idx].strip()
        letters = ["A", "B", "C", "D", "E", "F"]
        lines = [f"{letters[i]}: {opt}" for i, opt in enumerate(options_obj) if i < len(letters)]
        return "\n".join(lines)
    elif isinstance(options_obj, str):
        return options_obj.strip()
    return ""

def get_option_text(options_obj, letter, cycle_idx=0):
    """
    Extracts the textual content of a given option letter (A, B, C, D) from options_obj.
    Supports dict schemas {'A': '...'}, list-of-cycles schemas ['A. text\nB. text', ...],
    plain list schemas ['text_A', 'text_B', ...], and raw strings.
    """
    if not letter or str(letter).strip().upper() in ["NONE", "NULL", "UNKNOWN", ""]:
        return "None"
    target = str(letter).strip().upper()
    if isinstance(options_obj, dict):
        return str(options_obj.get(target, options_obj.get(target.lower(), ""))).strip()
    elif isinstance(options_obj, list):
        if len(options_obj) > 0 and isinstance(options_obj[0], str) and ('\n' in options_obj[0] or 'A.' in options_obj[0] or 'A:' in options_obj[0]):
            idx = cycle_idx if (0 <= cycle_idx < len(options_obj)) else 0
            block = options_obj[idx]
            m = re.search(rf'^{target}[\.\:\s]+(.+)$', block, flags=re.MULTILINE)
            if m:
                return m.group(1).strip()
            return ""
        letters = ["A", "B", "C", "D", "E", "F"]
        if target in letters:
            idx = letters.index(target)
            if idx < len(options_obj):
                return str(options_obj[idx]).strip()
    elif isinstance(options_obj, str):
        m = re.search(rf'^{target}[\.\:\s]+(.+)$', options_obj, flags=re.MULTILINE)
        if m:
            return m.group(1).strip()
    return ""

def resolve_gt_letter(answer_obj, cycle_idx=0):
    if isinstance(answer_obj, (list, tuple)):
        idx = cycle_idx if (0 <= cycle_idx < len(answer_obj)) else 0
        val = str(answer_obj[idx]).strip().upper() if len(answer_obj) > 0 else 'A'
        m = re.search(r'\b([A-D])\b', val)
        return m.group(1) if m else (val[0] if val and val[0] in 'ABCD' else 'A')
    if isinstance(answer_obj, str):
        clean = answer_obj.strip()
        if clean.startswith('[') and clean.endswith(']'):
            try:
                import ast
                parsed = ast.literal_eval(clean)
                if isinstance(parsed, (list, tuple)) and len(parsed) > 0:
                    idx = cycle_idx if (0 <= cycle_idx < len(parsed)) else 0
                    val = str(parsed[idx]).strip().upper()
                    m = re.search(r'\b([A-D])\b', val)
                    return m.group(1) if m else (val[0] if val and val[0] in 'ABCD' else 'A')
            except Exception:
                pass
            letters = re.findall(r'[A-Da-d]', clean)
            if letters:
                idx = cycle_idx if (0 <= cycle_idx < len(letters)) else 0
                return letters[idx].upper()
        m = re.search(r'\b([A-D])\b', clean.upper())
        if m:
            return m.group(1)
        for ch in clean.upper():
            if ch in ['A', 'B', 'C', 'D']:
                return ch
    return 'A'

def judge_sample(question, options_obj, prediction, cycle_idx=0, max_retries=2):
    options_str = format_options(options_obj, cycle_idx=cycle_idx)
    user_msg = DOUBLE_BLIND_USER_TEMPLATE.format(
        question=question,
        options_formatted=options_str,
        prediction=prediction
    )
    
    messages = [
        {"role": "system", "content": DOUBLE_BLIND_SYSTEM_PROMPT},
        {"role": "user", "content": user_msg}
    ]
    
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
    
    for attempt in range(max_retries + 1):
        with torch.no_grad():
            generated_ids = model.generate(
                **model_inputs,
                max_new_tokens=128,
                temperature=0.01,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )
        
        prompt_len = model_inputs.input_ids.shape[1]
        response_ids = generated_ids[0][prompt_len:]
        raw_reply = tokenizer.decode(response_ids, skip_special_tokens=True).strip()
        
        # Parse JSON reply
        try:
            # Clean markdown code block fences if present
            cleaned = re.sub(r"^```(?:json)?\s*", "", raw_reply, flags=re.IGNORECASE)
            cleaned = re.sub(r"\s*```$", "", cleaned)
            parsed = json.loads(cleaned)
            chosen = str(parsed.get("chosen_option", "")).strip().upper()
            if chosen in ["A", "B", "C", "D"]:
                return chosen, parsed.get("confidence", 1.0), parsed.get("reasoning", ""), raw_reply
            elif chosen in ["NONE", "Z", "NULL", "UNKNOWN"]:
                return "None", 0.0, parsed.get("reasoning", ""), raw_reply
        except Exception:
            # Regex fallback extraction
            m = re.search(r'"chosen_option"\s*:\s*"([A-Da-d]|None)"', raw_reply)
            if m:
                opt = m.group(1).upper()
                return ("None" if opt == "NONE" else opt), 0.8, "Regex fallback parse", raw_reply
                
    return "None", 0.0, "Parsing failure", raw_reply

print("[SUCCESS] Double-blind evaluation engine and prompt templates verified!")

In [ ]:
print("=" * 75)
print("EXECUTING LLM JUDGE EVALUATION ON PREDICTIONS")
print("=" * 75)

if not os.path.exists(PREDICTIONS_INPUT_PATH):
    raise FileNotFoundError(f"Predictions input file not found: {PREDICTIONS_INPUT_PATH}")

predictions_data = []
with open(PREDICTIONS_INPUT_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            try:
                predictions_data.append(json.loads(line))
            except Exception:
                pass

print(f"Loaded {len(predictions_data)} prediction records from {PREDICTIONS_INPUT_PATH}")

# Load existing judged results if any
judged_indices = set()
judged_records = []
if os.path.exists(JUDGED_OUTPUT_JSONL):
    with open(JUDGED_OUTPUT_JSONL, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                try:
                    r = json.loads(line)
                    judged_records.append(r)
                    if 'index' in r and r['index'] is not None:
                        judged_indices.add(r['index'])
                except Exception:
                    pass
    print(f"[Resume] Found {len(judged_indices)} already-judged records in {JUDGED_OUTPUT_JSONL}")

new_judged = 0
with open(JUDGED_OUTPUT_JSONL, "a", encoding="utf-8") as out_f:
    for item in tqdm(predictions_data, desc="Judging Samples"):
        item_idx = item.get('index')
        if item_idx in judged_indices:
            continue
            
        question = item.get('question', '')
        options = item.get('options', {})
        cycle_idx = item.get('cycle_category', 0)
        try:
            cycle_idx = int(cycle_idx)
        except Exception:
            cycle_idx = 0
        pred_text = str(item.get('mssearch_prediction', item.get('prediction', ''))).strip()
        gt_raw = item.get('gt_letter', item.get('answer', item.get('ground_truth', '')))
        gt_answer = resolve_gt_letter(gt_raw, cycle_idx=cycle_idx)
        category = item.get('category', 'single')
        
        # Perform Double-Blind Evaluation
        chosen, conf, reason, raw_r = judge_sample(question, options, pred_text, cycle_idx=cycle_idx)
        
        # Determine correctness
        is_correct = (chosen == gt_answer) if chosen in ["A", "B", "C", "D"] else False
        
        record = dict(item)
        record["judge_chosen_option"] = chosen
        record["judge_confidence"] = conf
        record["judge_reasoning"] = reason
        record["is_correct"] = is_correct
        record["gt_answer"] = gt_answer
        
        out_f.write(json.dumps(record, ensure_ascii=False) + "\n")
        out_f.flush()
        os.fsync(out_f.fileno())
        
        judged_records.append(record)
        judged_indices.add(item_idx)
        new_judged += 1

print(f"\n[JUDGE COMPLETE] Evaluated {new_judged} new samples. Total judged: {len(judged_records)}")


In [ ]:
print("=" * 75)
print("BENCHMARK ACCURACY REPORT: HRBENCH 8K")
print("=" * 75)

if not judged_records:
    print("No judged records available to calculate metrics.")
else:
    df = pd.DataFrame(judged_records)
    total_count = len(df)
    overall_acc = df["is_correct"].mean() * 100.0
    
    # Category breakdown
    breakdown = []
    for cat, group in df.groupby("category"):
        cat_acc = group["is_correct"].mean() * 100.0
        breakdown.append({
            "Category": cat,
            "Total Samples": len(group),
            "Correct": int(group["is_correct"].sum()),
            "Accuracy (%)": f"{cat_acc:.2f}%"
        })
    
    breakdown.append({
        "Category": "OVERALL (All Categories)",
        "Total Samples": total_count,
        "Correct": int(df["is_correct"].sum()),
        "Accuracy (%)": f"{overall_acc:.2f}%"
    })
    
    summary_df = pd.DataFrame(breakdown)
    print(summary_df.to_string(index=False))
    print("=" * 75)
    
    # Route distribution
    if 'route' in df:
        print("\nAccuracy by MSSearch Execution Route:")
        route_df = df.groupby('route').agg(
            Samples=('is_correct', 'count'),
            Accuracy=('is_correct', lambda x: f"{x.mean()*100:.2f}%")
        ).reset_index()
        print(route_df.to_string(index=False))
        
    # Export Markdown Summary Report
    report_path = os.path.join(OUTPUT_DIR, "HRBENCH8K_ACCURACY_REPORT.md")
    with open(report_path, "w", encoding="utf-8") as f:
        f.write("# HRBench 8K Benchmark Accuracy Report\n\n")
        f.write(f"- **Total Samples Evaluated**: {total_count}\n")
        f.write(f"- **Overall Accuracy**: **{overall_acc:.2f}%**\n")
        f.write(f"- **Judge Model**: `{JUDGE_MODEL_ID}` (Double-Blind Protocol)\n\n")
        f.write("## Category Breakdown\n\n")
        f.write(summary_df.to_markdown(index=False) + "\n\n")
    print(f"\n[REPORT] Saved markdown summary to: {report_path}")


In [ ]:
print("=" * 75)
print("ERROR & CONFLICT CASE AUDIT & EXPORT")
print("=" * 75)

if judged_records:
    wrong_samples = []
    
    for r in judged_records:
        if not r.get("is_correct", False):
            options = r.get("options", {})
            cycle_idx = r.get("cycle_category", 0)
            try:
                cycle_idx = int(cycle_idx)
            except Exception:
                cycle_idx = 0
                
            gt_letter = r.get("gt_answer", r.get("gt_letter", ""))
            chosen_opt = r.get("judge_chosen_option", "None")
            
            gt_text = get_option_text(options, gt_letter, cycle_idx=cycle_idx)
            chosen_text = get_option_text(options, chosen_opt, cycle_idx=cycle_idx)
            
            # High-level error categorization
            if chosen_opt in ["None", "null", "UNKNOWN"]:
                error_type = "no_option_matched"
            elif chosen_opt in ["A", "B", "C", "D"]:
                error_type = "option_mismatch"
            else:
                error_type = "judge_parsing_error"
                
            wrong_entry = {
                "index": r.get("index"),
                "category": r.get("category", "single"),
                "cycle_category": cycle_idx,
                "question": r.get("question", ""),
                "options": options,
                "ground_truth_letter": gt_letter,
                "ground_truth_text": gt_text,
                "model_prediction": r.get("mssearch_prediction", r.get("prediction", "")),
                "raw_vlm_response": r.get("raw_vlm_response", ""),
                "extracted_answers": r.get("extracted_answers", []),
                "judge_chosen_option": chosen_opt,
                "judge_chosen_text": chosen_text,
                "judge_confidence": r.get("judge_confidence", 0.0),
                "judge_reasoning": r.get("judge_reasoning", ""),
                "error_type": error_type,
                "route": r.get("route", "unknown"),
                "latency_sec": r.get("latency_sec", 0.0),
                "evidence_images": r.get("evidence_images", []),
                "input_image": r.get("input_image", "")
            }
            wrong_samples.append(wrong_entry)
            
    total_judged = len(judged_records)
    total_errors = len(wrong_samples)
    error_rate = (total_errors / total_judged * 100.0) if total_judged > 0 else 0.0
    
    print(f"Total Evaluated Samples : {total_judged}")
    print(f"Total Incorrect Samples : {total_errors} ({error_rate:.2f}% error rate)\n")
    
    # Save to JSONL
    with open(WRONG_SAMPLES_JSONL, "w", encoding="utf-8") as f:
        for entry in wrong_samples:
            f.write(json.dumps(entry, ensure_ascii=False) + "\n")
    print(f"[EXPORT] Saved {len(wrong_samples)} wrong samples to JSONL: {WRONG_SAMPLES_JSONL}")
    
    # Save to formatted JSON
    with open(WRONG_SAMPLES_JSON, "w", encoding="utf-8") as f:
        json.dump(wrong_samples, f, ensure_ascii=False, indent=2)
    print(f"[EXPORT] Saved {len(wrong_samples)} wrong samples to formatted JSON: {WRONG_SAMPLES_JSON}\n")
    
    # Distribution of error types
    err_counts = Counter(e["error_type"] for e in wrong_samples)
    print("Error Type Breakdown:")
    for et, count in err_counts.most_common():
        print(f"  • {et:20s}: {count:4d} ({count / total_errors * 100:.1f}%)")
    print()

    # Display audit preview for the first 5 errors
    print("--- FIRST 5 INCORRECT SAMPLES AUDIT PREVIEW ---")
    for i, err in enumerate(wrong_samples[:5]):
        print(f"\n[Error #{i+1}] Index {err['index']} | Category: {err['category']} | Error Type: {err['error_type']} | Route: {err['route']}")
        print(f"Question     : {err['question']}")
        print(f"Ground Truth : Option {err['ground_truth_letter']} -> \"{err['ground_truth_text']}\"")
        print(f"Model Pred   : \"{err['model_prediction']}\"")
        print(f"Judge Chosen : Option {err['judge_chosen_option']} -> \"{err['judge_chosen_text']}\" (Conf: {err['judge_confidence']})")
        print(f"Reasoning    : {err['judge_reasoning']}")

print("=" * 75)